# 09 - Create Data Splits (Stratified by Signer)

Splits the dataset into train/val/test sets, keeping all clips from a given signer in exactly one set.

**Strategy: Stratified Group Split by Signer ID**
- Split is based on `signer_id` — no signer appears in more than one set.
- Split is stratified by `label` to preserve class distribution across sets.
- Two-step `StratifiedGroupKFold`: first 80/20 (train vs hold-out), then 50/50 on the hold-out (val vs test).
- Optionally augments the training set with 12 spatial/temporal transforms.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
plt.style.use('ggplot')

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

INPUT_CSV     = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_videos_normalized.csv'
OUTPUT_CSV    = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_with_splits.csv'
AUG_OUTPUT_DIR = PROJECT_ROOT / 'normalized_videos' / 'topn_keepfps_224_gray_augmented'

N_SPLITS_TRAIN_TEST = 5
N_SPLITS_VAL_TEST   = 2
RANDOM_SEED = 42

# Set to True to augment training data with 12 spatial/temporal transforms.
# Val and test are never augmented.
AUGMENT_TRAIN = True

MAX_WORKERS = 12
OVERWRITE   = False

AUGMENTATIONS = [
    {"name": "slow_x0_8",     "kind": "temporal_resample", "factor": 0.8},
    {"name": "fast_x1_25",    "kind": "temporal_resample", "factor": 1.25},
    {"name": "interp_plus20", "kind": "temporal_resample", "factor": 1.0 / 1.2},
    {"name": "drop_every_5",  "kind": "drop_stride",       "stride": 5},
    {"name": "shift_left_8",  "kind": "shift",  "dx": -8, "dy":  0},
    {"name": "shift_right_8", "kind": "shift",  "dx":  8, "dy":  0},
    {"name": "shift_up_8",    "kind": "shift",  "dx":  0, "dy": -8},
    {"name": "shift_down_8",  "kind": "shift",  "dx":  0, "dy":  8},
    {"name": "rot_left_5",    "kind": "rotate", "angle": -5.0},
    {"name": "rot_right_5",   "kind": "rotate", "angle":  5.0},
    {"name": "zoom_in_110",   "kind": "zoom",   "scale": 1.10},
    {"name": "zoom_out_90",   "kind": "zoom",   "scale": 0.90},
]

print('INPUT_CSV :', INPUT_CSV)
print('OUTPUT_CSV:', OUTPUT_CSV)
print('AUGMENT_TRAIN:', AUGMENT_TRAIN)

INPUT_CSV : C:\Users\Magda\source\repos\private\szum\merged_datasets\universal_metadata_topn_videos_normalized.csv
OUTPUT_CSV: C:\Users\Magda\source\repos\private\szum\merged_datasets\universal_metadata_topn_with_splits.csv
AUGMENT_TRAIN: True


## Load and Prepare Data

In [3]:
df_all = pd.read_csv(INPUT_CSV)
print('Total rows:', len(df_all))
print(df_all['process_status'].value_counts(dropna=False).to_string())

df = df_all[df_all['process_status'] == 'ok'].copy()
df = df.dropna(subset=['signer_id']).copy()
df['signer_id'] = df['signer_id'].astype(int)

print(f'\nRows after filtering: {len(df)}')
print(f'Unique labels:  {df["label"].nunique()}')
print(f'Unique signers: {df["signer_id"].nunique()}')

Total rows: 614
process_status
ok    614

Rows after filtering: 614
Unique labels:  50
Unique signers: 60


## Perform the Stratified Group Split

In [4]:
X      = df
y      = df['label']
groups = df['signer_id']

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS_TRAIN_TEST, shuffle=True, random_state=RANDOM_SEED)
train_idx, temp_idx = next(sgkf.split(X, y, groups))

df['split'] = ''
df.iloc[train_idx, df.columns.get_loc('split')] = 'train'
df.iloc[temp_idx,  df.columns.get_loc('split')] = 'temp'

temp_df = df[df['split'] == 'temp']
sgkf2   = StratifiedGroupKFold(n_splits=N_SPLITS_VAL_TEST, shuffle=True, random_state=RANDOM_SEED)
val_rel, test_rel = next(sgkf2.split(temp_df, temp_df['label'], temp_df['signer_id']))

df.loc[temp_df.index[val_rel],  'split'] = 'val'
df.loc[temp_df.index[test_rel], 'split'] = 'test'

df['is_augmented']     = False
df['augmentation_name'] = 'original'

print(df['split'].value_counts().to_string())

split
train    483
test      78
val       53


c:\Users\Magda\source\repos\private\szum\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1037: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


## Ensure Every Label Appears in Train and at Least One of Val/Test

In [ ]:
# For each label present only in train, try to move one signer to val or test.
# Only possible if the label has >= 2 signers (signer separation must be preserved).
# Signer with the fewest clips is moved to whichever of val/test is currently smaller.

moved = 0
for label in sorted(df['label'].unique()):
    label_df = df[df['label'] == label]
    in_val_or_test = label_df['split'].isin(['val', 'test']).any()
    if in_val_or_test:
        continue

    signers = label_df.groupby('signer_id').size().sort_values()
    if len(signers) < 2:
        continue  # only one signer for this label, cannot split

    donor_signer = signers.index[0]
    target_split = 'val' if len(df[df['split'] == 'val']) <= len(df[df['split'] == 'test']) else 'test'

    mask = (df['label'] == label) & (df['signer_id'] == donor_signer)
    df.loc[mask, 'split'] = target_split
    moved += 1
    print(f'  [{label}] signer {donor_signer} ({int(signers[donor_signer])} clips) -> {target_split}')

print(f'\nLabels fixed: {moved}')
print(df['split'].value_counts().to_string())

## Verify the Splits

In [5]:
def safe_fps(row: pd.Series) -> float:
    for key in ['normalized_fps', 'fps', 'input_fps']:
        try:
            f = float(row.get(key, float('nan')))
            if f > 1e-6:
                return f
        except (TypeError, ValueError):
            pass
    return 30.0


def to_repo_path(p: Path) -> str:
    try:
        return str(p.relative_to(PROJECT_ROOT)).replace('\\', '/')
    except ValueError:
        return str(p).replace('\\', '/')


def read_frames(video_path: Path) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()
    return frames


def write_video(frames: list[np.ndarray], output_path: Path, fps: float) -> bool:
    if not frames:
        return False
    h, w = frames[0].shape[:2]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    if not writer.isOpened():
        return False
    for f in frames:
        writer.write(f)
    writer.release()
    return True


def temporal_resample(frames: list[np.ndarray], factor: float) -> list[np.ndarray]:
    n_in  = len(frames)
    n_out = max(2, int(round(n_in / factor)))
    out   = []
    for i in range(n_out):
        pos = i * (n_in - 1) / (n_out - 1)
        l   = int(np.floor(pos))
        r   = min(l + 1, n_in - 1)
        a   = float(pos - l)
        out.append(frames[l] if r == l else cv2.addWeighted(frames[l], 1 - a, frames[r], a, 0))
    return out


def drop_stride(frames: list[np.ndarray], stride: int) -> list[np.ndarray]:
    out = [f for i, f in enumerate(frames) if (i + 1) % stride != 0]
    return out if out else [frames[0]]


def apply_augmentation(frames: list[np.ndarray], aug: dict[str, Any]) -> list[np.ndarray]:
    kind = aug['kind']
    if kind == 'temporal_resample':
        return temporal_resample(frames, float(aug['factor']))
    if kind == 'drop_stride':
        return drop_stride(frames, int(aug['stride']))
    if kind == 'shift':
        m = np.float32([[1, 0, aug['dx']], [0, 1, aug['dy']]])
        h, w = frames[0].shape[:2]
        return [cv2.warpAffine(f, m, (w, h), borderMode=cv2.BORDER_REPLICATE) for f in frames]
    if kind == 'rotate':
        h, w = frames[0].shape[:2]
        m = cv2.getRotationMatrix2D((w / 2, h / 2), float(aug['angle']), 1.0)
        return [cv2.warpAffine(f, m, (w, h), borderMode=cv2.BORDER_REPLICATE) for f in frames]
    if kind == 'zoom':
        scale = float(aug['scale'])
        result = []
        for f in frames:
            h, w  = f.shape[:2]
            nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
            resized = cv2.resize(f, (nw, nh))
            if scale >= 1.0:
                y0, x0 = (nh - h) // 2, (nw - w) // 2
                result.append(resized[y0:y0 + h, x0:x0 + w])
            else:
                canvas = np.zeros_like(f)
                y0, x0 = (h - nh) // 2, (w - nw) // 2
                canvas[y0:y0 + nh, x0:x0 + nw] = resized
                result.append(canvas)
        return result
    raise ValueError(f'Unknown augmentation kind: {kind}')


def augment_row(row: pd.Series) -> list[dict[str, Any]]:
    src_path = PROJECT_ROOT / str(row['normalized_video_path'])
    fps      = safe_fps(row)
    results  = []

    frames = read_frames(src_path)
    if not frames:
        return results

    for aug in AUGMENTATIONS:
        aug_name = aug['name']
        out_path = AUG_OUTPUT_DIR / str(row['label']) / f"{src_path.stem}__{aug_name}.mp4"

        aug_row = row.to_dict()
        aug_row['normalized_video_path'] = to_repo_path(out_path)
        aug_row['is_augmented']          = True
        aug_row['augmentation_name']     = aug_name
        aug_row['split']                 = 'train'

        if not out_path.exists() or OVERWRITE:
            try:
                write_video(apply_augmentation(frames, aug), out_path, fps)
            except Exception as exc:
                print(f'  augmentation error {src_path.name} / {aug_name}: {exc}')

        results.append(aug_row)
    return results

In [6]:
if AUGMENT_TRAIN:
    AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_df = df[df['split'] == 'train']
    print(f'Augmenting {len(train_df)} training clips x {len(AUGMENTATIONS)} transforms ...')

    aug_records: list[dict] = []
    done, total = 0, len(train_df)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(augment_row, row): idx for idx, row in train_df.iterrows()}
        for future in as_completed(futures):
            aug_records.extend(future.result())
            done += 1
            if done % 25 == 0 or done == total:
                print(f'  {done}/{total}')

    aug_df   = pd.DataFrame(aug_records)
    df_final = pd.concat([df, aug_df], ignore_index=True)
    print(f'\nDone. {len(aug_df)} augmented rows added.')
else:
    df_final = df.copy()
    print('Augmentation skipped (AUGMENT_TRAIN=False).')

print(df_final.groupby(['split', 'is_augmented']).size().to_string())

Augmenting 483 training clips x 12 transforms ...


  25/483
  50/483
  75/483
  100/483
  125/483
  150/483
  175/483
  200/483
  225/483
  250/483
  275/483
  300/483
  325/483
  350/483
  375/483
  400/483
  425/483
  450/483
  475/483
  483/483

Done. 5796 augmented rows added.
split  is_augmented
test   False             78
train  False            483
       True            5796
val    False             53


In [7]:
base = df[df['is_augmented'] == False]

split_stats = base.groupby('split').agg(
    n_samples=('label', 'count'),
    n_signers=('signer_id', 'nunique'),
)
split_stats['pct'] = (split_stats['n_samples'] / len(base) * 100).round(1)

print('Split statistics (original clips only):')
display(split_stats)

train_signers = set(base[base['split'] == 'train']['signer_id'])
val_signers   = set(base[base['split'] == 'val'  ]['signer_id'])
test_signers  = set(base[base['split'] == 'test' ]['signer_id'])

overlap = {
    'train/val' : len(train_signers & val_signers),
    'train/test': len(train_signers & test_signers),
    'val/test'  : len(val_signers   & test_signers),
}
print('\nSigner overlap:')
for pair, count in overlap.items():
    status = 'OK' if count == 0 else 'OVERLAP DETECTED'
    print(f'  {pair}: {count}  [{status}]')

Split statistics (original clips only):


,n_samples,n_signers,pct
split,,,
test,78,6,12.7
train,483,47,78.7
val,53,7,8.6



Signer overlap:
  train/val: 0  [OK]
  train/test: 0  [OK]
  val/test: 0  [OK]


## Save

In [8]:
df_final = df_final.sort_values(['label', 'split', 'is_augmented', 'signer_id']).reset_index(drop=True)
df_final.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
print(df_final.groupby(['split', 'is_augmented']).size().rename('rows').to_string())

Saved: C:\Users\Magda\source\repos\private\szum\merged_datasets\universal_metadata_topn_with_splits.csv
split  is_augmented
test   False             78
train  False            483
       True            5796
val    False             53
